In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# Configuration
MODEL_NAME = "Salesforce/codet5p-220m"   # 🔁 Replaced CodeBERT with CodeT5+
NUM_LABELS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def load_and_preprocess_data(base_path, tokenizer, folder_name):
    codes = []
    labels = []
    for label_dir in ["Label_0", "Label_1"]:
        current_path = os.path.join(base_path, folder_name, label_dir)
        if not os.path.exists(current_path):
            print(f"Warning: Directory {current_path} not found. Skipping.")
            continue
        for filename in os.listdir(current_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(current_path, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    codes.append(f.read())
                labels.append(0 if label_dir == "Label_0" else 1)

    # --- Tokenize normally ---
    encodings = tokenizer(
        codes,
        truncation=True,
        padding="max_length",
        max_length=512,
        add_special_tokens=True,
        return_tensors="pt"
    )

    # --- Fix: ensure exactly one <eos> token per example ---
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id

    fixed_input_ids = []
    for ids in encodings["input_ids"]:
        ids = ids.tolist()

        # Remove all existing eos tokens
        ids = [tid for tid in ids if tid != eos_id]
        # Add exactly one eos token at end
        ids.append(eos_id)

        # Re-pad or truncate to 512
        if len(ids) < 512:
            ids += [pad_id] * (512 - len(ids))
        else:
            ids = ids[:512]

        fixed_input_ids.append(ids)

    encodings["input_ids"] = torch.tensor(fixed_input_ids)

    return CodeDataset(encodings, labels)


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

base_text_path = "../Text_Files/"

# Use CodeT5+ model instead of RoBERTa
MODEL_NAME = "Salesforce/codet5p-220m"
NUM_LABELS = 2

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

model.eval()


Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at Salesforce/codet5p-220m and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


T5ForSequenceClassification(
  (transformer): T5Model(
    (shared): Embedding(32100, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32100, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=768, out_features=3072, bias=False)
                (wo): Linear(in_feat

In [5]:
# Load and preprocess training data
print("Loading training & validation data...")
train_dataset = load_and_preprocess_data(base_text_path, tokenizer, "Train")

Loading training & validation data...


In [6]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
)

In [8]:
# Trainer 
trainer = Trainer( model=model, args=training_args, train_dataset=train_dataset, ) 
print("Training model...") 
trainer.train()

Training model...


/tmp/ipykernel_715329/2307531119.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss
10,0.727600
20,0.881000
30,0.816100
40,0.838000
50,0.866600
60,0.717700
70,0.896300
80,0.727900
90,0.740400
100,0.761800


/tmp/ipykernel_715329/2307531119.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipykernel_715329/2307531119.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


TrainOutput(global_step=2322, training_loss=0.3060046417469602, metrics={'train_runtime': 1377.9058, 'train_samples_per_second': 13.477, 'train_steps_per_second': 1.685, 'total_flos': 1.13421272914944e+16, 'train_loss': 0.3060046417469602, 'epoch': 3.0})

In [7]:
import os
current_dir = os.getcwd()
print(current_dir)

/home/info-sec-lab/BTP/SO/experiments


In [26]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # make CUDA synchronous (set BEFORE any CUDA ops)

In [8]:
model_path = "../checkpoints/ct5p_only/checkpoint-2322"
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True, num_labels=NUM_LABELS)

# Load tokenizer from the original pre-trained model (not from checkpoint)
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")  # or whatever base model you used

# Create trainer with the loaded model
trainer = Trainer(model=model)

In [11]:
from collections import Counter
cnt = Counter()

for i in range(min(len(test_dataset), 1000000)):
    ids = test_dataset[i]["input_ids"]
    cnt[int((ids == tokenizer.eos_token_id).sum())] += 1

print(cnt)

Counter({1: 1001, 2: 1})


/tmp/ipykernel_728651/2307531119.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


## 🧪 Testing on Checkpoint 3 — `CodeT5+`


In [14]:
# Evaluate on test datasets
print("Evaluating on test datasets...")
for i in range(10):
    test_folder = f"Test_{i}"
    print(f"Loading test data for {test_folder}...")
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    if len(test_dataset) > 0:
        predictions = trainer.predict(test_dataset)
        # Process predictions to get labels
        predicted_labels = predictions.predictions[0].argmax(axis=1)
        true_labels = test_dataset.labels

        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

        print(f"Results for {test_folder}:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")


Evaluating on test datasets...
Loading test data for Test_0...


/tmp/ipykernel_728651/97132969.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Results for Test_0:
  Accuracy: 0.8992
  Precision: 0.9329
  Recall: 0.8603
  F1-Score: 0.8951
Loading test data for Test_1...


/tmp/ipykernel_728651/97132969.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Results for Test_1:
  Accuracy: 0.7605
  Precision: 0.7171
  Recall: 0.8603
  F1-Score: 0.7822
Loading test data for Test_2...


...

Results for Test_7:
  Accuracy: 0.6377
  Precision: 0.5953
  Recall: 0.8603
  F1-Score: 0.7037
Loading test data for Test_8...


/tmp/ipykernel_728651/97132969.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Results for Test_8:
  Accuracy: 0.6936
  Precision: 0.6452
  Recall: 0.8603
  F1-Score: 0.7374
Loading test data for Test_9...


/tmp/ipykernel_728651/97132969.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Results for Test_9:
  Accuracy: 0.6377
  Precision: 0.5953
  Recall: 0.8603
  F1-Score: 0.7037
